### Benchmar using Lasso and Ridge Regression

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.feature_selection import RFE
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

In [3]:
train_set = pd.read_csv("data/train_normal_rtf.csv")
test_set = pd.read_csv("data/test_normal_rtf.csv")

In [3]:
def generateSubBM(features, BM):
    n, _ = BM.shape
    subBM = np.zeros((n, len(features)), dtype=float)
    for idx, value in enumerate(features):
        subBM[:, idx] = BM[:, value].reshape(n,)
    return subBM

In [4]:
X_train = train_set.drop(columns=["label"]).values
y_train = train_set["label"].values
X_test = test_set.drop(columns=["label"]).values
y_test = test_set["label"].values

In [27]:
def get_metrics(y_predict, y_probability, y_test):
    tn, fp, fn, tp = metrics.confusion_matrix(y_test, y_predict).ravel()
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    mse = metrics.mean_squared_error(y_test, y_predict)
    #tn, fp, fn, tp
    auc, f1, mcc = metrics.roc_auc_score(
        y_test,  y_probability[:, 1]), metrics.f1_score(y_test, y_predict), metrics.matthews_corrcoef(y_test, y_predict)
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "mse": mse,
        "auc": auc,
        "f1_score": f1,
        "mcc": mcc,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

##### Logistic regression with L1 regularization (Lasso) is used for feature selection. https://scikit-learn.org/stable/modules/feature_selection.html

In [28]:
stratifiedKFold = RepeatedStratifiedKFold(
    n_splits=10, n_repeats=3, random_state=45)

# Use these lists to store metrics for the comparison table
lasso_metrics = []
svm_rfe_metrics = []

# Your existing StratifiedKFold loop
for train_index, test_index in stratifiedKFold.split(X_train, y_train):
    # Split using the indices from your KFold object
    X_train_fold, X_test_fold = X_train[train_index], X_train[test_index]
    y_train_fold, y_test_fold = y_train[train_index], y_train[test_index]

    # --- 1. BASELINE: LASSO (Logistic Regression with L1) ---
    # liblinear is required for L1 penalty in LogisticRegression
    # C=1.0 is the inverse of regularization strength; smaller values = more sparsity
    lasso = LogisticRegression(penalty='l1', solver='liblinear', random_state=45, max_iter=1000)
    lasso.fit(X_train_fold, y_train_fold)
    
    y_pred_lasso = lasso.predict(X_test_fold)
    y_proba = lasso.predict_proba(X_test_fold)
    lasso_metrics.append(get_metrics(y_pred_lasso, y_proba, y_test_fold))

    # --- 2. BASELINE: SVM-RFE ---
    # We use a linear kernel to rank the importance of all columns
    # step=0.1 speeds up the process by removing 10% of genes per iteration
    svc_lin = SVC(kernel="linear", probability=True, random_state=45)
    rfe = RFE(estimator=svc_lin, n_features_to_select=10, step=0.1)
    rfe.fit(X_train_fold, y_train_fold)
    
    y_pred_svm = rfe.predict(X_test_fold)
    y_proba_svm = rfe.predict_proba(X_test_fold)
    svm_rfe_metrics.append(get_metrics(y_pred_svm, y_proba_svm, y_test_fold))

In [19]:
len(lasso_metrics)

30

In [29]:
lasso_metrics

[{'accuracy': 0.75,
  'precision': 0.7272727272727273,
  'recall': 0.8421052631578947,
  'specificity': 0.6470588235294118,
  'mse': 0.25,
  'auc': 0.8018575851393188,
  'f1_score': 0.7804878048780488,
  'mcc': 0.5009339514249774,
  'tn': 11,
  'fp': 6,
  'fn': 3,
  'tp': 16},
 {'accuracy': 0.6666666666666666,
  'precision': 0.6842105263157895,
  'recall': 0.6842105263157895,
  'specificity': 0.6470588235294118,
  'mse': 0.3333333333333333,
  'auc': 0.6749226006191951,
  'f1_score': 0.6842105263157895,
  'mcc': 0.33126934984520123,
  'tn': 11,
  'fp': 6,
  'fn': 6,
  'tp': 13},
 {'accuracy': 0.75,
  'precision': 0.7368421052631579,
  'recall': 0.7777777777777778,
  'specificity': 0.7222222222222222,
  'mse': 0.25,
  'auc': 0.824074074074074,
  'f1_score': 0.7567567567567567,
  'mcc': 0.5007733956671915,
  'tn': 13,
  'fp': 5,
  'fn': 4,
  'tp': 14},
 {'accuracy': 0.7222222222222222,
  'precision': 0.7222222222222222,
  'recall': 0.7222222222222222,
  'specificity': 0.7222222222222222,


In [31]:
# Mean and standard deviation across CV folds for each metric
lasso_df = pd.DataFrame(lasso_metrics)
svm_rfe_df = pd.DataFrame(svm_rfe_metrics)

metrics_summary = pd.DataFrame({
    "lasso_mean": lasso_df.mean(),
    "lasso_std": lasso_df.std(),
    "svm_rfe_mean": svm_rfe_df.mean(),
    "svm_rfe_std": svm_rfe_df.std(),
}).round(4)

print(metrics_summary)

             lasso_mean  lasso_std  svm_rfe_mean  svm_rfe_std
accuracy         0.6995     0.0717        0.6930       0.0643
precision        0.6929     0.0677        0.6780       0.0608
recall           0.7453     0.1133        0.7694       0.1109
specificity      0.6515     0.0961        0.6133       0.0985
mse              0.3005     0.0717        0.3070       0.0643
auc              0.7581     0.0727        0.7378       0.0725
f1_score         0.7146     0.0767        0.7170       0.0666
mcc              0.4041     0.1444        0.3951       0.1336
tn              11.3333     1.7087       10.6667       1.7287
fp               6.0667     1.7207        6.7333       1.7604
fn               4.6333     2.0592        4.2000       2.0410
tp              13.5667     2.0957       14.0000       2.0342
